# OER Workflow via Explog + JobHandler to OH Values

This notebook demonstrates an end-to-end workflow using `Explog` and `Job_handler` in `mode="test"`.
It runs bulk, surface, O-adsorption, and OH-adsorption via process submission
to obtain OH-related values.

In [67]:
from pathlib import Path
import sys
import shutil

import pandas as pd
from ase.io import read

In [68]:
# Ensure local package import works when running from notebooks/
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from gnome_dreams_oer_screening import Explog, OER_catalyst_study

In [69]:
# Load test bulk structure
bulk_atoms = read(repo_root / "notebooks" / "TiO2_rutile.cif")
bulk_atoms

Atoms(symbols='Ti2O4', pbc=True, cell=[4.59983732, 4.59983732, 2.95921356], spacegroup_kinds=...)

In [70]:
# Initialize EXPLOG and add one candidate
work_dir = repo_root / "notebooks" / "TEMP_explog_oer_workflow"
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

EXPLOG = Explog()
EXPLOG.init(work_dir, mode="test", prefix_vasp_dirs=False)

candidate_id = "demo_candidate_1"
study = OER_catalyst_study(
    bulk_atoms.copy(),
    H2O_gas_free_energy=-14.217,
    H2_gas_free_energy=-6.77,
)
EXPLOG.add_candidate(candidate_id=candidate_id, study_obj=study)

EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,G(O) deviation,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,<NA>,<NA>,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,<NA>,<NA>


In [71]:
# Run bulk relaxation via process API
p_bulk = EXPLOG.add_process(candidate_id=candidate_id, calculation_type="bulk_relaxation")
EXPLOG.submit_process(p_bulk)
updates = EXPLOG.update_log()
print("updates:", updates)

cand_study = EXPLOG.relational_frame.candidates[candidate_id].study_obj
print("bulk set:", cand_study.get_bulk_atoms() is not None)

updates: {'0': 'completed'}
bulk set: True


In [72]:
# Rank surfaces, initialize one termination, and run surface relaxation via process API
cand_study.predict_most_likely_surfaces(max_miller=1, method="O-coordination")
ranking = cand_study.get_termination_rankings().sort_values("Normalized score", ascending=False)
termination_index = int(ranking.iloc[0]["Termination index"])
cand_study.initialize_oer_surface_study(termination_index)

p_surface = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="surface_relaxation",
    termination_index=termination_index,
)
EXPLOG.submit_process(p_surface)
updates = EXPLOG.update_log()
print("updates:", updates)
ranking

updates: {'1': 'completed'}


,Miller indices,Termination index,Score,Normalized score,Surface atoms:
2,"(1, 1, 0)",2,-0.051948,1.000000,"[Ti, Ti]"
4,"(1, 0, 0)",4,-0.073465,0.593465,[Ti]
3,"(1, 0, 1)",3,-0.079495,0.479541,"[Ti, Ti]"
5,"(0, 0, 1)",5,-0.094525,0.195581,"[Ti, Ti]"
1,"(1, 1, 1)",1,-0.104877,0.000000,"[Ti, Ti]"
0,"(1, 1, 1)",0,-0.104877,0.000000,[Ti]


In [73]:
# Determine adsorption sites and initialize one adsorption-site study
surface_study = cand_study.get_surface_studies()[termination_index]
surface_study.determine_adsorption_sites(only_reduced_coord_O_sites=True)
sites_df = surface_study.get_adsorption_sites_df()

if (sites_df["site type"] == "on-top").any():
    site_index = int(sites_df[sites_df["site type"] == "on-top"]["Site index"].iloc[0])
else:
    site_index = int(sites_df["Site index"].iloc[0])

surface_study.initialize_adsorption_site_study(site_index)
sites_df[["Site index", "site type", "ad site element"]]

,Site index,site type,ad site element
0,0,on-top,Ti
1,1,on-top,Ti
2,2,lattice O,N/A


In [74]:
# Run O adsorption via process API
p_o = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="O_adsorption",
    termination_index=termination_index,
    site_index=site_index,
)
EXPLOG.submit_process(p_o)
updates = EXPLOG.update_log()
print("updates:", updates)
EXPLOG.relational_frame.processes.df.tail(3)

updates: {'2': 'completed'}


,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,demo_candidate_1,bulk_relaxation,42,completed,<NA>,<NA>,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>
1,1,demo_candidate_1,surface_relaxation,43,completed,2,<NA>,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>
2,2,demo_candidate_1,O_adsorption,44,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>


In [75]:
sites_df = surface_study.get_adsorption_sites_df()
sites_df

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G(O),G(O) deviation,G(OH),G(OH) deviation,G(OOH) from scaling relation,"G(OOH, scaling) deviation",ideal overpotential,overpotential from OH-OOH scaling relation
0,0,on-top,Ti,"[(O, 1.9518096083479362), (O, 1.95180960834793...",N/A,"[1.4796067800000001, 4.270518461424494, 24.937...",6.0,1.238223,1.221777,None,None,None,None,None,None
1,1,on-top,Ti,"[(O, 1.951809608347936), (O, 1.951809608347936...",N/A,"[-3.311838709094295e-17, 1.0811051314143296, 2...",7.0,None,None,None,None,None,None,None,None
2,2,lattice O,N/A,"[(Ti, 1.951809608347936), (Ti, 1.9518096083479...",1,"[1.47960678, 0.8875227549160024, 24.6073687959...",23.0,None,None,None,None,None,None,None,None


In [76]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,G(O) deviation,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,<NA>,<NA>,<gnome_dreams_oer_screening.oer.oer_study.OER_...,1.22,<NA>,<NA>


In [77]:
# Run OH adsorption via process API
p_oh_list = EXPLOG.add_process(
    candidate_id=candidate_id,
    calculation_type="OH_adsorption",
    termination_index=termination_index,
    site_index=site_index,
)
for p_oh in p_oh_list:
    EXPLOG.submit_process(p_oh)
updates = EXPLOG.update_log()
print("updates:", updates)
EXPLOG.relational_frame.processes.df.tail(6)

updates: {'3': 'completed', '4': 'completed', '5': 'completed'}


,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,demo_candidate_1,bulk_relaxation,42,completed,<NA>,<NA>,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>
1,1,demo_candidate_1,surface_relaxation,43,completed,2,<NA>,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>
2,2,demo_candidate_1,O_adsorption,44,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,<NA>
3,3,demo_candidate_1,OH_adsorption,45,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=0
4,4,demo_candidate_1,OH_adsorption,46,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=1
5,5,demo_candidate_1,OH_adsorption,47,completed,2,0,c:\Users\matnis\OneDrive - Danmarks Tekniske U...,OH_init_index=2


In [78]:
# Update derived energies/overpotentials and inspect outputs
surface_study.update_adsorption_energies()

final_df = surface_study.get_adsorption_sites_df()
cols = [
    c for c in [
        "Site index",
        "site type",
        "G(O)",
        "G(OH)",
        "ideal overpotential",
        "overpotential from OH-OOH scaling relation",
    ]
    if c in final_df.columns
]
display(final_df.loc[[site_index], cols])

candidate_row = EXPLOG.relational_frame.candidates[candidate_id]
print("idealOverPotential:", candidate_row.idealOverPotential)
print("Overpotential_from_scaling:", candidate_row.Overpotential_from_scaling)
EXPLOG.relational_frame.candidates.df

,Site index,site type,G(O),G(OH),ideal overpotential,overpotential from OH-OOH scaling relation
0,0,on-top,1.238223,-1.168594,1.18,1.66


idealOverPotential: 1.18
Overpotential_from_scaling: 1.66


,candidate_id,reason_or_hypothesis,notes,study_obj,G(O) deviation,Overpotential_from_scaling,idealOverPotential
0,demo_candidate_1,<NA>,<NA>,<gnome_dreams_oer_screening.oer.oer_study.OER_...,1.22,1.66,1.18


In [79]:
sites_df = surface_study.get_adsorption_sites_df()
sites_df

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G(O),G(O) deviation,G(OH),G(OH) deviation,G(OOH) from scaling relation,"G(OOH, scaling) deviation",ideal overpotential,overpotential from OH-OOH scaling relation
0,0,on-top,Ti,"[(O, 1.9518096083479362), (O, 1.95180960834793...",N/A,"[1.4796067800000001, 4.270518461424494, 24.937...",6.0,1.238223,1.221777,-1.168594,2.398594,2.031406,1.658594,1.18,1.66
1,1,on-top,Ti,"[(O, 1.951809608347936), (O, 1.951809608347936...",N/A,"[-3.311838709094295e-17, 1.0811051314143296, 2...",7.0,None,None,None,None,None,None,None,None
2,2,lattice O,N/A,"[(Ti, 1.951809608347936), (Ti, 1.9518096083479...",1,"[1.47960678, 0.8875227549160024, 24.6073687959...",23.0,None,None,None,None,None,None,None,None
